# ARC-v0.16.1a — FEVER Initial-Feature Provenance Equality Audit

This audit resolves one provenance question from ARC-v0.16.1:

> Are the `v013_initial_query_features.parquet` artifacts associated with the frozen ARC-v0.13 run (`20260817-140640`) and the later run (`20260817-151852`) identical in content?

The audit checks:

- file existence and SHA-256,
- schema,
- row count and unique query coverage,
- duplicate consistency,
- exact query-id set equality,
- exact / numerical equality of `pq32_entropy20` and `pq32_margin1_10`,
- and writes a sealed report.

If the two artifacts are identical by value, the paper can safely state that ARC-v0.16.1 recovered an equivalent persisted query-feature artifact. If they differ, the ablation should be rerun using the frozen `20260817-140640` artifact.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

from google.colab import drive

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Drive mount failed"

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
V013_ROOT = ARC_ROOT / "fever-boundary-external-replication-v013"

FROZEN_RUN = V013_ROOT / "20260817-140640"
LATER_RUN = V013_ROOT / "20260817-151852"

assert FROZEN_RUN.is_dir(), FROZEN_RUN
assert LATER_RUN.is_dir(), LATER_RUN

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

print("Frozen run:", FROZEN_RUN)
print("Later run :", LATER_RUN)


In [ ]:
# Locate candidate feature artifacts in both runs.

def find_feature_candidates(run):
    candidates = []

    for p in run.rglob("*"):
        if not p.is_file():
            continue

        if p.suffix.lower() not in {".parquet", ".csv"}:
            continue

        name = p.name.lower()

        if (
            "initial_query_feature" in name
            or "query_feature" in name
            or "baseline_feature" in name
        ):
            candidates.append(p)

    return sorted(candidates)

frozen_candidates = find_feature_candidates(FROZEN_RUN)
later_candidates = find_feature_candidates(LATER_RUN)

print("Frozen candidates:")
for p in frozen_candidates:
    print(" ", p)

print("\nLater candidates:")
for p in later_candidates:
    print(" ", p)

assert frozen_candidates, "No feature artifact found in frozen run"
assert later_candidates, "No feature artifact found in later run"

# Prefer the canonical filename when present.
def choose_candidate(candidates):
    exact = [
        p for p in candidates
        if p.name == "v013_initial_query_features.parquet"
    ]
    return exact[0] if exact else candidates[0]

FROZEN_FEATURES = choose_candidate(frozen_candidates)
LATER_FEATURES = choose_candidate(later_candidates)

print("\nSelected frozen:", FROZEN_FEATURES)
print("Selected later :", LATER_FEATURES)


In [ ]:
required = [
    "query_id",
    "pq32_entropy20",
    "pq32_margin1_10",
]

def load_feature_table(path):
    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)

    assert set(required).issubset(df.columns), (
        path,
        df.columns.tolist(),
    )

    out = df[required].copy()
    out["query_id"] = out["query_id"].astype(str)

    return out

frozen_raw = load_feature_table(FROZEN_FEATURES)
later_raw = load_feature_table(LATER_FEATURES)

print("Frozen shape:", frozen_raw.shape)
print("Later shape :", later_raw.shape)

print("Frozen unique queries:", frozen_raw["query_id"].nunique())
print("Later unique queries :", later_raw["query_id"].nunique())

print("Frozen SHA:", sha256_file(FROZEN_FEATURES))
print("Later SHA :", sha256_file(LATER_FEATURES))


In [ ]:
# Duplicate consistency audit.

def duplicate_consistency(df, label):
    g = (
        df.groupby("query_id")[
            ["pq32_entropy20", "pq32_margin1_10"]
        ]
        .nunique(dropna=False)
    )

    bad = g[
        (g["pq32_entropy20"] > 1)
        | (g["pq32_margin1_10"] > 1)
    ]

    print(label, "inconsistent duplicate queries:", len(bad))

    assert len(bad) == 0, (
        label,
        bad.head(20),
    )

duplicate_consistency(frozen_raw, "frozen")
duplicate_consistency(later_raw, "later")

frozen = (
    frozen_raw
    .drop_duplicates("query_id")
    .sort_values("query_id")
    .reset_index(drop=True)
)

later = (
    later_raw
    .drop_duplicates("query_id")
    .sort_values("query_id")
    .reset_index(drop=True)
)

assert frozen["query_id"].is_unique
assert later["query_id"].is_unique

print("DEDUP CONSISTENCY — PASS")


In [ ]:
# Query-id set equality.

frozen_ids = set(frozen["query_id"])
later_ids = set(later["query_id"])

only_frozen = sorted(frozen_ids - later_ids)
only_later = sorted(later_ids - frozen_ids)

print("Only frozen:", len(only_frozen))
print("Only later :", len(only_later))

if only_frozen:
    print("Example frozen-only:", only_frozen[:20])

if only_later:
    print("Example later-only:", only_later[:20])

query_id_sets_equal = (
    len(only_frozen) == 0
    and len(only_later) == 0
)

print("Query-id sets equal:", query_id_sets_equal)


In [ ]:
# Value equality audit on matched query ids.

matched = frozen.merge(
    later,
    on="query_id",
    how="inner",
    suffixes=("_frozen", "_later"),
    validate="one_to_one",
)

print("Matched queries:", len(matched))

results = {}

for col in [
    "pq32_entropy20",
    "pq32_margin1_10",
]:
    a = matched[f"{col}_frozen"].to_numpy(np.float64)
    b = matched[f"{col}_later"].to_numpy(np.float64)

    finite_equal = np.array_equal(
        np.isfinite(a),
        np.isfinite(b),
    )

    exact_equal = np.array_equal(a, b)

    abs_diff = np.abs(a - b)

    allclose_1e12 = np.allclose(
        a,
        b,
        rtol=0.0,
        atol=1e-12,
        equal_nan=True,
    )

    allclose_1e9 = np.allclose(
        a,
        b,
        rtol=0.0,
        atol=1e-9,
        equal_nan=True,
    )

    differing = int(
        np.sum(
            ~np.isclose(
                a,
                b,
                rtol=0.0,
                atol=1e-12,
                equal_nan=True,
            )
        )
    )

    results[col] = {
        "finite_mask_equal": bool(finite_equal),
        "exact_equal": bool(exact_equal),
        "allclose_atol_1e-12": bool(allclose_1e12),
        "allclose_atol_1e-9": bool(allclose_1e9),
        "max_abs_diff": float(np.nanmax(abs_diff)) if len(abs_diff) else 0.0,
        "mean_abs_diff": float(np.nanmean(abs_diff)) if len(abs_diff) else 0.0,
        "differing_rows_at_1e-12": differing,
    }

    print("\n", col)
    print(json.dumps(results[col], indent=2))

content_equal_1e12 = (
    query_id_sets_equal
    and all(
        results[c]["allclose_atol_1e-12"]
        for c in results
    )
)

print("\nCONTENT EQUAL @ 1e-12:", content_equal_1e12)


In [ ]:
# Show worst differences if any.

diff_view = matched.copy()

diff_view["entropy_abs_diff"] = np.abs(
    diff_view["pq32_entropy20_frozen"]
    - diff_view["pq32_entropy20_later"]
)

diff_view["margin_abs_diff"] = np.abs(
    diff_view["pq32_margin1_10_frozen"]
    - diff_view["pq32_margin1_10_later"]
)

worst = (
    diff_view
    .sort_values(
        ["entropy_abs_diff", "margin_abs_diff"],
        ascending=False,
    )
    .head(20)
)

display(
    worst[
        [
            "query_id",
            "pq32_entropy20_frozen",
            "pq32_entropy20_later",
            "entropy_abs_diff",
            "pq32_margin1_10_frozen",
            "pq32_margin1_10_later",
            "margin_abs_diff",
        ]
    ]
)


In [ ]:
# Seal report.

OUT_ROOT = ARC_ROOT / "feature-provenance-equality-audit-v0161a"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

report = {
    "status": "ARC_V0161A_FEATURE_PROVENANCE_EQUALITY_AUDIT_COMPLETE",
    "frozen_run": str(FROZEN_RUN),
    "later_run": str(LATER_RUN),
    "frozen_feature_artifact": str(FROZEN_FEATURES),
    "later_feature_artifact": str(LATER_FEATURES),
    "frozen_sha256": sha256_file(FROZEN_FEATURES),
    "later_sha256": sha256_file(LATER_FEATURES),
    "frozen_rows_raw": int(len(frozen_raw)),
    "later_rows_raw": int(len(later_raw)),
    "frozen_unique_queries": int(frozen["query_id"].nunique()),
    "later_unique_queries": int(later["query_id"].nunique()),
    "query_id_sets_equal": bool(query_id_sets_equal),
    "column_comparisons": results,
    "content_equal_atol_1e-12": bool(content_equal_1e12),
    "recommended_source_for_paper": (
        str(FROZEN_FEATURES)
        if content_equal_1e12
        else "RERUN_ABLATION_FROM_FROZEN_FEATURE_ARTIFACT"
    ),
    "test_accessed": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = OUT / "v0161a_feature_provenance_equality_report.json"
REPORT_PATH.write_text(
    json.dumps(report, indent=2, sort_keys=True),
    encoding="utf-8",
)

report_sha = sha256_file(REPORT_PATH)

(OUT / "V0161A_REPORT_SHA256.txt").write_text(
    report_sha + "  " + REPORT_PATH.name + "\n",
    encoding="utf-8",
)

print("=" * 80)
print("ARC-v0.16.1a FEATURE PROVENANCE AUDIT — COMPLETE")
print("=" * 80)
print("Content equal @ 1e-12:", content_equal_1e12)
print("Frozen artifact SHA:", report["frozen_sha256"])
print("Later artifact SHA :", report["later_sha256"])
print("Report:", REPORT_PATH)
print("Report SHA:", report_sha)

if content_equal_1e12:
    print("\nDECISION: VALUE-EQUIVALENT — keep ARC-v0.16.1 results.")
else:
    print("\nDECISION: NOT EQUIVALENT — rerun ARC-v0.16.1 from frozen artifact.")
